# Composition — Example

We model a `Notifier` that needs to send messages through different channels. Instead of subclassing per channel, `Notifier` **holds** a `MessageSender` object and **delegates** the actual sending to it, so channels can be swapped freely, even at runtime.

In [ ]:
from typing import Protocol


class MessageSender(Protocol):
    def send(self, text: str) -> None: ...


class EmailSender:
    def send(self, text: str) -> None:
        print(f"[email] {text}")


class SmsSender:
    def send(self, text: str) -> None:
        print(f"[sms] {text}")

In [ ]:
class Notifier:
    """Has-a MessageSender; delegates delivery instead of inheriting a channel-specific class."""

    def __init__(self, sender: MessageSender):
        self._sender = sender

    def notify(self, text: str) -> None:
        self._sender.send(f"Notification: {text}")


email_notifier = Notifier(EmailSender())
sms_notifier = Notifier(SmsSender())

email_notifier.notify("Your order has shipped")
sms_notifier.notify("Your order has shipped")

## Swapping behavior at runtime

Because `Notifier` only depends on the small `MessageSender` interface, we can change which sender it uses after the object already exists — something inheritance can't do (an object's class is fixed once created).

In [ ]:
notifier = Notifier(EmailSender())
notifier.notify("Using email for now")

notifier._sender = SmsSender()  # swap the composed dependency at runtime
notifier.notify("Switched to SMS")

## Easier testing via injected fakes

Because the sender is injected, we can substitute a fake for tests without touching `Notifier` or building a subclass hierarchy.

In [ ]:
class FakeSender:
    def __init__(self):
        self.sent: list[str] = []

    def send(self, text: str) -> None:
        self.sent.append(text)


fake = FakeSender()
test_notifier = Notifier(fake)
test_notifier.notify("test message")

assert fake.sent == ["Notification: test message"]
print("Test passed:", fake.sent)

## Reflection

- `Notifier` never inherits from `EmailSender` or `SmsSender` — it only holds a reference and delegates via the shared `MessageSender` protocol.
- Adding a new channel (e.g., `PushSender`) requires zero changes to `Notifier`.
- Swapping `_sender` at runtime and injecting a `FakeSender` for testing are both only possible because the dependency is composed, not inherited.